# ActivIQ: auto-labelling + LoRA pipeline
Runs the rest of section 4.6 of the proposal, in order:

1. **APE**: Qwen3.5-9B writes candidate labelling prompts from the 10 seeds; each is scored on the 100 validation rows and compared with the team baseline prompt.
2. **Label**: the winning prompt labels the 960 training utterances (machine labels).
3. **LoRA**: Qwen3.5-2B is fine-tuned on those machine labels.
4. **Compare**: LoRA 2B vs DistilBERT + spaCy (and the 9B zero-shot run) on the 120 human-labelled test utterances.

Use a **GPU runtime** (A100/L4 best; a T4 works but the 9B stages are slow). All results are saved to Google Drive, so after a disconnect you can re-run the cells from the top: finished work is reused (`--resume`).
Never use the test split for prompt selection or training.

In [ ]:
# 1. Drive + repo. Results live on Drive (reports/ is a symlink to it), so they survive a disconnect.
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE = "/content/drive/MyDrive/ActivIQ_runs"
os.makedirs(DRIVE, exist_ok=True)

REPO, BRANCH = "jiakai2002/ActivIQ", "lora-finetuning"
# Don't push code changes between sessions of the same run: resume is refused if the experiment code changed.
!rm -rf /content/ActivIQ && git clone -q --depth 1 -b {BRANCH} https://github.com/{REPO}.git /content/ActivIQ
%cd /content/ActivIQ
!ln -sfn {DRIVE} reports
!git log --oneline -1

In [ ]:
# 2. Dependencies (keep Colab's own torch)
!pip install -q "transformers>=5.3,<6" "peft>=0.13,<1" "accelerate>=1,<2" safetensors bitsandbytes pandas spacy
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# 9B always 4-bit. 2B: full precision on bf16 GPUs (L4/A100), 4-bit on a T4 so LoRA weights train in fp32.
Q9B = "4bit"
Q2B = "none" if BF16 else "4bit"
print("9B quantization:", Q9B, "| 2B quantization:", Q2B)

In [ ]:
# 3. Preflight (no model): checks the human-reviewed seeds and validation labels.
!python scripts/auto_label.py check | head -20
print("\n=== prompts/auto_labelling/baseline.txt ===")
print(open("prompts/auto_labelling/baseline.txt").read())

### Before running `score`: freeze the baseline prompt
The baseline above was written by an assistant. The proposal compares APE against *your team's own prompt*. Read it, edit `prompts/auto_labelling/baseline.txt` in the repo if needed, push, and re-run from cell 1. Then set `PROMPT_FROZEN = True` in the scoring cell. (`propose` does not use the baseline, so you can run it first.)

In [ ]:
# 4. Download models to local Colab disk (not Drive). ~19 GB + ~5 GB, a few minutes each session.
from huggingface_hub import snapshot_download
for repo, d in [("Qwen/Qwen3.5-9B", "models/qwen3.5-9b"), ("Qwen/Qwen3.5-2B", "models/qwen3.5-2b")]:
    snapshot_download(repo, local_dir=d); print("ok", d)

In [ ]:
# 4b. Zero-shot 9B benchmark on the 120 test utterances (skipped if results are already on Drive).
# If you ran the old benchmark notebook, copy its qwen9b_zeroshot folder to MyDrive/ActivIQ_runs/evaluation/ to skip this.
import os
if os.path.exists("reports/evaluation/qwen9b_zeroshot/lora_comparison.json"):
    print("zero-shot results already exist, skipping")
else:
    !python scripts/evaluate_lora.py --model-path models/qwen3.5-9b --device cuda --quantization {Q9B} \
        --skip-baseline --output-dir reports/evaluation/qwen9b_zeroshot


In [ ]:
# 5. APE step 1: the 9B model proposes candidate prompts from the 10 seeds.
!python scripts/auto_label.py propose --model-path models/qwen3.5-9b --device cuda --quantization {Q9B} \
    --candidate-count 4 --run-dir reports/auto_labelling/propose --resume

In [ ]:
# 6. APE step 2: score the baseline + candidates on the 100 validation rows.
PROMPT_FROZEN = False   # set True after you have reviewed/frozen prompts/auto_labelling/baseline.txt
assert PROMPT_FROZEN, "Review and freeze the baseline prompt first (see the note above)."
!python scripts/auto_label.py score --model-path models/qwen3.5-9b --device cuda --quantization {Q9B} \
    --candidates reports/auto_labelling/propose/candidates.jsonl --run-dir reports/auto_labelling/score --resume

In [ ]:
# 7. Which prompt won? (validation selection scores, NOT held-out performance)
import json, pandas as pd
sel = json.load(open("reports/auto_labelling/score/selection.json"))
print("Selected prompt:", sel["selected_prompt"]["id"], "| entity target met:", sel["validation_entity_target_met"])
display(pd.DataFrame([{"prompt": r["id"], "entity_micro_f1": r["metrics"]["micro"]["f1"],
    "valid_output_rate": r["metrics"]["valid_output_rate"], "intent_agreement": r["metrics"]["intent_agreement"]}
    for r in sel["ranking"]]))
print("Targets: entity F1 >= 0.80, intent agreement >= 0.85 (proposal section 5)")


In [ ]:
# 8. Label the 960 training utterances with the selected prompt (the long step; safe to resume).
!python scripts/auto_label.py label --model-path models/qwen3.5-9b --device cuda --quantization {Q9B} \
    --selection reports/auto_labelling/score/selection.json --run-dir reports/auto_labelling/train_labels --resume

In [ ]:
# 9. Label quality: how often does the 9B intent match the dataset's source intent?
import json
rows = [json.loads(l) for l in open("reports/auto_labelling/train_labels/machine_labels.jsonl")]
rej = sum(1 for _ in open("reports/auto_labelling/train_labels/rejected.jsonl")) if os.path.exists("reports/auto_labelling/train_labels/rejected.jsonl") else 0
agree = sum(r["intent"] == r["source_intent"] for r in rows) / len(rows)
print(f"accepted {len(rows)} | rejected {rej} | intent agreement {agree:.3f} (target >= 0.85)")

In [ ]:
# 10. LoRA fine-tune Qwen3.5-2B on the machine labels (new output dir each time).
!python scripts/train_lora.py --model-path models/qwen3.5-2b --device cuda --quantization {Q2B} \
    --labels reports/auto_labelling/train_labels/machine_labels.jsonl --output-dir reports/lora/adapter_v1

### DistilBERT baseline in Colab (optional but recommended)
So both systems run on the same machine, zip your local `models/intent_classifier/` folder (about 255 MB), upload it to `MyDrive/ActivIQ_runs/intent_classifier.zip`, then run the next cell. Without it, the comparison skips DistilBERT and you must report its numbers from your laptop (note the different hardware for latency).

In [ ]:
# 11. Restore DistilBERT if you uploaded it.
import os
z = f"{DRIVE}/intent_classifier.zip"
HAVE_BASELINE = os.path.exists(z)
if HAVE_BASELINE:
    !mkdir -p models && unzip -qo {z} -d models/
    !ls models/intent_classifier | head
print("baseline available:", HAVE_BASELINE)

In [ ]:
# 12. Evaluate the LoRA 2B (and DistilBERT + spaCy if available) on the 120 human-labelled test utterances.
skip = "" if HAVE_BASELINE else "--skip-baseline"
!python scripts/evaluate_lora.py --model-path models/qwen3.5-2b --adapter reports/lora/adapter_v1 \
    --device cuda --quantization {Q2B} {skip} --output-dir reports/evaluation/lora_2b

In [ ]:
# 13. Final comparison table.
import json, os, pandas as pd
def row(name, r):
    e = r["entities"]; f1 = e["micro"]["f1"] if "micro" in e else e.get("status")
    return {"system": name, "intent_acc": r["intent"]["accuracy"], "intent_macro_f1": r["intent"]["macro_f1"],
            "entity_micro_f1": f1, "median_latency_s": r["latency_seconds"].get("median_seconds"),
            "invalid_outputs": r.get("invalid_outputs", 0)}
rep = json.load(open("reports/evaluation/lora_2b/lora_comparison.json"))
table = []
if "distilbert_spacy" in rep: table.append(row("DistilBERT + spaCy", rep["distilbert_spacy"]))
table.append(row("Qwen3.5-2B + LoRA (machine labels)", rep["lora"]))
zs = "reports/evaluation/qwen9b_zeroshot/lora_comparison.json"
if os.path.exists(zs): table.append(row("Qwen3.5-9B zero-shot", json.load(open(zs))["lora"]))
pd.set_option("display.float_format", "{:.3f}".format)
display(pd.DataFrame(table))
print("Latency: GPU for Qwen, CPU/GPU per your runtime for DistilBERT; state the hardware in the report.")

In [ ]:
# 14. Everything is already on Drive under ActivIQ_runs/. Optionally download a zip too.
!zip -qr results.zip reports/auto_labelling/score reports/auto_labelling/train_labels/summary.md reports/lora/adapter_v1/training_report.json reports/evaluation
from google.colab import files; files.download("results.zip")